# Evaluate topic analogies

For **A : B :: C : D**, rank topics by cosine similarity to **B − A + C** in the normalized embedding space. A, B, and C are excluded; D is only the expected answer.

Run all cells to evaluate every row in `evaluation/topic_analogy_candidates.csv` and see the final top-5, top-3, and top-1 counts. The same evaluator powers the standalone check and batch results. Ties preserve embedding-file order. Validation failures are reported and count as non-hits in the overall summary.

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("my_data/sgns/topic_embeddings_normalized.csv")
ANALOGY_PATH = Path("evaluation/topic_analogy_candidates.csv")
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]

# Load topic embeddings
topics = pd.read_csv(DATA_PATH, keep_default_na=False, dtype={"Topic_Id": str})
embedding_columns = sorted(c for c in topics.columns if c.startswith("embedding_"))
vectors = topics[embedding_columns].to_numpy(dtype=np.float64)
print(f"Embedding file: {DATA_PATH.resolve()}")
print(f"Loaded {len(topics):,} topics × {len(embedding_columns)} dimensions.")

# Create topic paths and a mapping from topic ID to row index
paths = topics[TOPIC_COLUMNS].apply(lambda row: "/".join(str(v) for v in row if v), axis=1)
id_to_index = {topic_id: i for i, topic_id in enumerate(topics["Topic_Id"])}

# Load analogies
analogies = pd.read_csv(ANALOGY_PATH, keep_default_na=False, dtype=str)
print(f"Loaded {len(analogies):,} analogies from {ANALOGY_PATH}.")


Embedding file: /Users/nachodoradollamas/Downloads/derStandard/my_data/sgns/topic_embeddings_normalized.csv
Loaded 1,000 topics × 150 dimensions.
Loaded 173 analogies from evaluation/topic_analogy_candidates.csv.


In [ ]:
def evaluate_analogy(analogy, top_k=5, verbose=False):
    """
    Accept an analogy row or ID and return (metrics, neighbors).
    Set verbose=True to print the analogy details and nearest topics.
    A:B::C:D analogy evaluation using cosine similarity of normalized embeddings.
    B-A+C should yield a vector similar to D. Ties follow embedding-file row order.
    An expected answer equal to A, B, or C is excluded and counts as a miss.
    """
    if not isinstance(top_k, int) or isinstance(top_k, bool) or top_k < 1:
        raise ValueError("top_k must be a positive integer.")
    if isinstance(analogy, str):
        analogy_id = analogy
        selected = analogies.loc[analogies["analogy_id"] == analogy_id]
        if len(selected) != 1:
            raise ValueError(f"Expected exactly one analogy with ID {analogy_id}; found {len(selected)}.")
        analogy = selected.iloc[0]

    # Validate that all four topic IDs exist and match the expected paths
    indices = {}
    for role in "abcd":
        topic_id = analogy[f"topic_id_{role}"]
        if topic_id not in id_to_index:
            raise ValueError(f"Topic {role.upper()} (ID {topic_id}) is missing from the embeddings.")
        index = id_to_index[topic_id]
        if paths.iloc[index] != analogy[f"topic_path_{role}"]:
            raise ValueError(f"Path mismatch for {role.upper()}: {paths.iloc[index]} != "
                             f"{analogy[f'topic_path_{role}']}")
        indices[role] = index

    # Compute the analogy query vector and its cosine similarity to all topics 
    query = vectors[indices["b"]] - vectors[indices["a"]] + vectors[indices["c"]]
    query_norm = np.linalg.norm(query)
    if query_norm <= np.finfo(float).eps:
        raise ValueError("The analogy query has zero length; cosine similarity is undefined.")
    scores = vectors @ (query / query_norm)
    excluded = {indices[role] for role in "abc"}
    candidates = np.array([i for i in range(len(topics)) if i not in excluded], dtype=int)
    if not len(candidates):
        raise ValueError("No eligible topics remain after excluding A, B, and C.")
    ranking = candidates[np.argsort(-scores[candidates], kind="stable")]
    top_indices = ranking[:top_k]
    expected_position = np.flatnonzero(ranking == indices["d"])
    expected_rank = int(expected_position[0]) + 1 if len(expected_position) else None
    metrics = {
        "analogy_id": analogy["analogy_id"],
        "relation": analogy["relation"],
        "label": analogy["label"],
        "expected_topic_id": analogy["topic_id_d"],
        "expected_path": paths.iloc[indices["d"]],
        "expected_rank": expected_rank,
        "expected_cosine_similarity": float(scores[indices["d"]]),
        "eligible_topics": len(candidates),
        "expected_excluded": indices["d"] in excluded,
        **{f"top{k}": expected_rank is not None and expected_rank <= k for k in (5, 3, 1)},
        "error": "",
    }
    neighbors = pd.DataFrame({
        "Rank": np.arange(1, len(top_indices) + 1),
        "Topic_Id": topics.iloc[top_indices]["Topic_Id"].to_numpy(),
        "Topic_path": paths.iloc[top_indices].to_numpy(),
        "Cosine_similarity": scores[top_indices],
        "Expected_answer": top_indices == indices["d"],
    })

    if verbose:
        print(f"{analogy['analogy_id']} | Relation: {analogy['relation']} | Label: {analogy['label']}")
        print(f"Notes: {analogy['notes'] or '(none)'}")
        for role in "abcd":
            purpose = "expected answer" if role == "d" else "query input"
            print(f"{role.upper()} ({purpose}): ID={analogy[f'topic_id_{role}']} | "
                  f"{analogy[f'topic_path_{role}']}")

        print("\n")
        print(neighbors.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
        
    return metrics, neighbors


def evaluate_all_analogies(analogy_rows):
    """Return one result per row, including validation errors instead of skipping them."""
    records = []
    for _, analogy in analogy_rows.iterrows():
        try:
            metrics, _ = evaluate_analogy(analogy)
        except ValueError as exc:
            metrics = {
                "analogy_id": analogy["analogy_id"],
                "relation": analogy["relation"],
                "label": analogy["label"],
                "expected_topic_id": analogy["topic_id_d"],
                "expected_path": analogy["topic_path_d"],
                "expected_rank": None,
                **{f"top{k}": False for k in (5, 3, 1)},
                "error": str(exc),
            }
        records.append(metrics)
    if not records:
        raise ValueError("No analogies to evaluate.")
    results = pd.DataFrame(records)
    results["expected_rank"] = results["expected_rank"].astype("Int64")
    return results


def summarize_analogies(results):
    """Count hits using all supplied rows as the denominator, including errors."""
    total = len(results)
    if not total:
        raise ValueError("No results to summarize.")
    return pd.DataFrame([
        {"Cutoff": f"Top {k}", "Matches": int(results[f"top{k}"].sum()),
         "Total analogies": total, "Hit rate (%)": 100 * results[f"top{k}"].sum() / total}
        for k in (5, 3, 1)
    ])


## Manually check one analogy

In [79]:
TOP_K = 5
for i in range(173, 174):
    ANALOGY_ID = f"AN{i:03d}"
    single_metrics, single_neighbors = evaluate_analogy(ANALOGY_ID, top_k=TOP_K, verbose=True)


AN173 | Relation: hierarchy_shortening | Label: structural
Notes: (none)
A (query input): ID=135 | wissenschaft/natur/klima
B (query input): ID=330 | wissenschaft/klima
C (query input): ID=487 | sport/mehrsport/schach
D (expected answer): ID=582 | sport/schach


 Rank Topic_Id                      Topic_path  Cosine_similarity  Expected_answer
    1      582                    sport/schach           0.710919             True
    2      756                  etat/sosindwir           0.498989            False
    3      144           sport/fussball/wm2022           0.476165            False
    4      241             web/webmix/webwatch           0.411510            False
    5      870 gesundheit/erkrankungentherapie           0.406079            False


## Evaluate every analogy

`all_results` contains expected-answer ranks, hit flags, and errors for each analogy. For example, inspect misses with `all_results.loc[~all_results["top5"]]`. The batch is independent of the standalone ID and neighbor count above.

In [4]:
all_results = evaluate_all_analogies(analogies)
errors = all_results.loc[all_results["error"].ne(""), ["analogy_id", "error"]]
print(f"Evaluated {len(all_results)} analogies: {len(all_results) - len(errors)} successful, {len(errors)} errors.")
if not errors.empty:
    print("\nValidation errors (included as non-hits):")
    print(errors.to_string(index=False))

summary = summarize_analogies(all_results)
print("\nExpected answers found in the top results:")
print(summary.to_string(index=False, float_format=lambda value: f"{value:.2f}"))

Evaluated 173 analogies: 173 successful, 0 errors.

Expected answers found in the top results:
Cutoff  Matches  Total analogies  Hit rate (%)
 Top 5      119              173         68.79
 Top 3      109              173         63.01
 Top 1       74              173         42.77
